<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/02_cnn_theory_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CNN을 처음부터 학습시켜보기 · MNIST 손글씨 숫자

**이 노트북은 선택 실습입니다. 채점·제출과 무관합니다.**

`00_quickstart_colab.ipynb`에서는 이미 학습된 ResNet18을 **빌려 썼습니다** (전이학습).
그래서 CNN 내부가 실제로 어떻게 학습되는지, epoch·optimizer·activation 같은 파라미터를
바꾸면 무엇이 달라지는지는 배터리 프로젝트 코드만으로는 직접 보기 어렵습니다.

이 노트북은 MNIST(0~9 손글씨 숫자, 28×28 흑백)로 아주 작은 CNN을 **처음부터** 학습시키며
그 부분을 직접 눈으로 확인합니다. 배터리 이미지(224×224 RGB) 대신 MNIST를 쓰는 이유는
간단합니다 — 다운로드가 즉시 끝나고, 한 epoch이 몇 초 안에 도니까 구조와 파라미터의 효과에만
집중할 수 있습니다.

**다루는 내용**
1. 데이터를 가져오고 전처리 단계를 하나씩 확인
2. CNN 구조(Conv → Pool → Dense) 정의
3. epoch / optimizer / activation을 바꿔가며 결과 비교
4. Confusion Matrix로 결과 해석
5. 이미지 한 장을 실제로 예측해보기

## 0. 환경 준비 (한글 폰트 + 라이브러리)

In [ ]:
# Colab 기본 matplotlib은 한글 폰트가 없어 그래프 제목이 네모(□)로 깨집니다.
# 한글이 필요없다면 이 셀은 건너뛰어도 됩니다 (그래프 제목만 영어로 바뀝니다).
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
import matplotlib.pyplot as plt
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

torch.manual_seed(42)  # 배터리 프로젝트와 같은 이유: 재현성 확보
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)  # cpu로 나오면 런타임 -> 런타임 유형 변경 -> T4 GPU

## 1. 데이터 가져오기

MNIST는 손글씨 숫자 0~9를 28×28 크기의 흑백(1채널) 이미지로 담은 데이터셋입니다.
torchvision이 처음 실행 시 자동으로 내려받습니다 (약 10초, 수십 MB).

In [ ]:
raw_train = datasets.MNIST(root='mnist_data', train=True, download=True)
raw_test = datasets.MNIST(root='mnist_data', train=False, download=True)
print('train 전체:', len(raw_train), '장')
print('test:', len(raw_test), '장')

### 1-1. 데이터 구조와 원본 이미지 직접 확인

In [ ]:
# 전처리를 거치기 전, 원본 데이터가 어떤 형태인지부터 확인합니다
img, label = raw_train[0]
print('타입:', type(img))          # PIL.Image.Image
print('크기(W,H):', img.size)      # (28, 28)
arr = np.array(img)
print('픽셀 값 범위:', arr.min(), '~', arr.max())  # 0~255, 아직 정규화 전
print('라벨:', label)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    im, lb = raw_train[i]
    ax.imshow(im, cmap='gray')
    ax.set_title(str(lb))
    ax.axis('off')
plt.suptitle('MNIST 원본 샘플 (전처리 전)')
plt.show()

### 1-2. 전처리 단계별로 직접 확인

배터리 프로젝트 슬라이드 13과 같은 흐름입니다. 다만 MNIST는 흑백 1채널이라 채널 수가
3이 아니라 1이고, 정규화 기준값도 ImageNet이 아니라 MNIST 자체의 평균·표준편차를 씁니다.

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize((0.1307,), (0.3081,))  # MNIST 학습 데이터 전체의 평균·표준편차

img, label = raw_train[0]

step1 = to_tensor(img)
print('1) ToTensor 후 shape:', tuple(step1.shape),
      '값 범위:', round(step1.min().item(), 3), '~', round(step1.max().item(), 3))
# [1, 28, 28] : 흑백 1채널, 0~255 픽셀을 0~1 범위로 변환

step2 = normalize(step1)
print('2) Normalize 후 값 범위:', round(step2.min().item(), 2), '~', round(step2.max().item(), 2))
# 평균 0·표준편차 1에 가깝게 재조정 -> 학습이 더 안정적으로 진행됨

## 2. train / valid / test 분할

MNIST는 이미 train 6만 장 / test 1만 장으로 나뉘어 배포됩니다.
여기서는 train 6만 장 중 1만 장을 다시 떼어 validation으로 씁니다
(배터리 프로젝트의 train/public_val 분할과 같은 역할입니다).

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), normalize])

train_full = datasets.MNIST(root='mnist_data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='mnist_data', train=False, download=True, transform=transform)

n_val = 10000
n_train = len(train_full) - n_val
train_set, val_set = random_split(
    train_full, [n_train, n_val], generator=torch.Generator().manual_seed(42))

print(f'train {len(train_set)}장 / val {len(val_set)}장 / test {len(test_set)}장')

## 3. CNN 구조 정의

Conv(필터로 지역 특징 추출) → 활성화함수 → MaxPool(크기 축소) 을 두 번 반복한 뒤,
Flatten으로 1차원으로 펼쳐 Dense(fc) 층에 통과시켜 최종 10개 클래스 점수를 냅니다.

- **Conv 층**: 작은 필터가 이미지 위를 이동하며 경계·곡선 같은 지역 패턴에 반응합니다. 필터 값 자체가 학습 대상입니다.
- **MaxPool 층**: 특징 맵을 절반 크기로 줄이며, 구역 안에서 가장 강하게 반응한 값만 남깁니다. 위치가 한두 픽셀 밀려도 같은 특징으로 인식하게 해줍니다 (파라미터가 없는 고정 연산).
- **Dense(fc) 층**: 지금까지 뽑힌 모든 특징을 한 줄로 펼쳐 연결한 뒤, 그것을 조합해 최종 점수로 바꿉니다.

`activation` 인자로 활성화 함수를 바꿔볼 수 있게 만들었습니다.

In [ ]:
ACTIVATIONS = {'relu': nn.ReLU, 'leaky_relu': nn.LeakyReLU, 'tanh': nn.Tanh, 'sigmoid': nn.Sigmoid}

class SimpleCNN(nn.Module):
    def __init__(self, activation='relu'):
        super().__init__()
        act = ACTIVATIONS[activation]
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1채널 입력 -> 필터 16개
        self.act1 = act()
        self.pool1 = nn.MaxPool2d(2, 2)                            # 28x28 -> 14x14
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 필터 16개 -> 32개
        self.act2 = act()
        self.pool2 = nn.MaxPool2d(2, 2)                            # 14x14 -> 7x7
        self.fc1 = nn.Linear(32 * 7 * 7, 64)                       # Dense: 특징을 64개로 압축
        self.act3 = act()
        self.fc2 = nn.Linear(64, 10)                               # Dense: 최종 10개 클래스 점수

    def forward(self, x):
        x = self.pool1(self.act1(self.conv1(x)))   # [1,28,28]  -> [16,14,14]
        x = self.pool2(self.act2(self.conv2(x)))    # [16,14,14] -> [32,7,7]
        x = x.view(x.size(0), -1)                     # 펼치기      -> [1568] (32*7*7)
        x = self.act3(self.fc1(x))                    # Dense       -> [64]
        return self.fc2(x)                              # Dense       -> [10]

In [ ]:
# 배치 하나를 실제로 넣어 텐서 모양이 정말 그렇게 바뀌는지 확인
sample_x, sample_y = next(iter(DataLoader(train_set, batch_size=4)))
probe_model = SimpleCNN()
out = probe_model(sample_x)
print('입력 :', tuple(sample_x.shape))   # [4, 1, 28, 28]
print('출력 :', tuple(out.shape))        # [4, 10]  배치 4장 x 클래스 10개 점수

## 4. 학습 함수

epoch / optimizer / activation / learning rate를 인자로 바꿀 수 있게 만들었습니다.
학습 루프 안의 4단계는 배터리 프로젝트 train_baseline.py, 슬라이드 17과 동일한 구조입니다.
각 실험은 같은 seed로 시작하므로 초기 가중치와 데이터 순서가 같고, 바꾼 옵션의 효과를 더 공정하게 비교할 수 있습니다.

In [ ]:
def build_optimizer(name, params, lr, weight_decay=0.0):
    """--optimizer 대응 함수. 배터리 프로젝트 train_baseline.py와 같은 선택지를 씁니다."""
    if name == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)   # 관성(momentum) 없이는 수렴이 느림
    if name == 'adam':
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay) # AdamW도 같은 정규화 강도로 비교
    raise ValueError(f'알 수 없는 optimizer: {name}')


@torch.no_grad()
def evaluate(model, loader):
    """validation/test 셋에 대해 accuracy, macro F1, confusion matrix를 계산합니다."""
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds.append(logits.argmax(1).cpu())
        trues.append(yb)
    preds = torch.cat(preds).numpy()
    trues = torch.cat(trues).numpy()
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average='macro')
    cm = confusion_matrix(trues, preds, labels=list(range(10)))
    return acc, f1, cm


def train_model(epochs=3, optimizer_name='adam', activation='relu', lr=1e-3,
                batch_size=128, seed=42, verbose=True):
    """주어진 설정 하나로 모델을 처음부터 학습시키고 (모델, history)를 돌려줍니다."""
    # 옵션의 효과만 비교하도록 매 실험의 초기 가중치와 데이터 순서를 같게 맞춥니다.
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = SimpleCNN(activation=activation).to(device)
    # optimizer 비교에서는 weight_decay를 모두 0으로 두어 변경 요인을 하나로 맞춥니다.
    optimizer = build_optimizer(optimizer_name, model.parameters(), lr, weight_decay=0.0)
    criterion = nn.CrossEntropyLoss()

    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, generator=generator)
    val_loader = DataLoader(val_set, batch_size=batch_size)

    history = {'train_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()                 # (1) 이전 배치의 gradient 초기화
            logits = model(xb)                     # (2) 예측
            loss = criterion(logits, yb)           # (3) 정답과의 차이 측정
            loss.backward()                        # (4) 역전파: 가중치별 책임 계산
            optimizer.step()                       # (5) 가중치 업데이트
            running += loss.item() * xb.size(0)
            seen += xb.size(0)

        train_loss = running / seen
        val_acc, val_f1, _ = evaluate(model, val_loader)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        if verbose:
            print(f'epoch {epoch}/{epochs}  train_loss {train_loss:.4f}  '
                  f'val_acc {val_acc:.4f}  val_f1 {val_f1:.4f}')

    return model, history

## 5. 실험 1 — epoch 수를 바꾸면 무엇이 달라지는가

같은 설정(optimizer=adam, activation=relu)에서 epoch만 1 / 3 / 5로 바꿔 비교합니다.
1 epoch은 전체 데이터를 한 번만 보고 멈춘 상태이므로 아직 충분히 배우지 못했을 가능성이 큽니다.
epoch이 늘어날수록 val_acc가 오르다가, 어느 지점부터는 정체되거나 떨어질 수 있습니다
— 2회차 자료의 overfitting 곡선과 같은 원리입니다.

In [ ]:
epoch_results = {}
for ep in [1, 3, 5]:
    print(f'--- epoch={ep} 학습 시작 ---')
    _, history = train_model(epochs=ep, optimizer_name='adam', activation='relu', verbose=False)
    epoch_results[f'epoch={ep}'] = history
    print(f'  최종 val_acc: {history["val_acc"][-1]:.4f}')

plt.figure(figsize=(6, 4))
for label, h in epoch_results.items():
    plt.plot(range(1, len(h['val_acc']) + 1), h['val_acc'], marker='o', label=label)
plt.xlabel('학습 진행 (epoch)')
plt.ylabel('validation accuracy')
plt.title('epoch 수에 따른 학습 곡선 비교')
plt.legend()
plt.show()

## 6. 실험 2 — optimizer를 바꾸면 무엇이 달라지는가

같은 3 epoch에서 optimizer만 sgd / adam / adamw로 바꿔 비교합니다.
일반적으로 SGD는 관성만으로 방향을 잡기 때문에 초반 수렴이 느리고, Adam 계열은
파라미터마다 학습 속도를 다르게 조절해 초반에 더 빠르게 손실을 줄이는 경향이 있습니다.

In [ ]:
opt_results = {}
for opt_name in ['sgd', 'adam', 'adamw']:
    _, history = train_model(epochs=3, optimizer_name=opt_name, activation='relu', verbose=False)
    opt_results[opt_name] = history
    print(f'{opt_name}: val_acc={history["val_acc"][-1]:.4f}')

plt.figure(figsize=(6, 4))
for name, h in opt_results.items():
    plt.plot(range(1, len(h['train_loss']) + 1), h['train_loss'], marker='o', label=name)
plt.xlabel('epoch')
plt.ylabel('train loss')
plt.title('optimizer에 따른 학습 손실 비교')
plt.legend()
plt.show()

## 7. 실험 3 — activation function을 바꾸면 무엇이 달라지는가

relu / leaky_relu / tanh / sigmoid를 비교합니다. sigmoid·tanh는 층이 깊어질수록
gradient가 0에 가까워지는 vanishing gradient 문제가 relu 계열보다 잘 발생합니다.
이 노트북의 CNN은 층이 얕아 차이가 크지 않을 수 있지만, 경향 자체는 확인할 수 있습니다.

In [ ]:
act_results = {}
for act_name in ['relu', 'leaky_relu', 'tanh', 'sigmoid']:
    _, history = train_model(epochs=3, optimizer_name='adam', activation=act_name, verbose=False)
    act_results[act_name] = history
    print(f'{act_name}: val_acc={history["val_acc"][-1]:.4f}')

plt.figure(figsize=(6, 4))
plt.bar(act_results.keys(), [h['val_acc'][-1] for h in act_results.values()])
plt.ylabel('validation accuracy (3 epoch 후)')
plt.title('activation function에 따른 최종 정확도 비교')
plt.show()

## 8. 최종 모델로 전체 결과 확인 (Confusion Matrix)

예시 설정(adam, relu, 5 epoch)으로 한 번 더 학습시킨 뒤
test set 전체에 대해 confusion matrix를 그립니다. 슬라이드 18에서 배운 것과 같은 방법으로,
행은 실제 숫자, 열은 예측한 숫자입니다.

In [ ]:
final_model, final_history = train_model(epochs=5, optimizer_name='adam', activation='relu')
test_loader = DataLoader(test_set, batch_size=256)
test_acc, test_f1, cm = evaluate(final_model, test_loader)
print(f'\nTest accuracy: {test_acc:.4f}   Test Macro F1: {test_f1:.4f}')

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel('예측 숫자')
plt.ylabel('실제 숫자')
for i in range(10):
    for j in range(10):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        plt.text(j, i, cm[i, j], ha='center', va='center', color=color, fontsize=8)
plt.title('Confusion Matrix (Test set)')
plt.colorbar()
plt.show()

## 9. 이미지 한 장 직접 예측해보기

test set에서 이미지 한 장을 뽑아, 모델이 실제로 어떤 확신도로 어떤 숫자라고 답하는지 확인합니다.

In [ ]:
sample_img, true_label = test_set[0]
model_input = sample_img.unsqueeze(0).to(device)   # 배치 차원 추가: [1,28,28] -> [1,1,28,28]

final_model.eval()
with torch.no_grad():
    logits = final_model(model_input)
    probs = F.softmax(logits, dim=1)
    pred_label = probs.argmax(1).item()

plt.imshow(sample_img.squeeze().cpu(), cmap='gray')
plt.title(f'실제: {true_label}  /  예측: {pred_label}  (확신도 {probs[0, pred_label]:.1%})')
plt.axis('off')
plt.show()

print('클래스별 확률:')
for i, p in enumerate(probs[0]):
    print(f'  {i}: {p:.4f}')

## 정리

- **Conv/Pool/Dense**로 이어지는 구조 자체는 배터리 프로젝트의 ResNet18과 원리가 같습니다.
  ResNet18은 이 구조가 훨씬 많이 쌓여 있고, ImageNet으로 이미 학습되어 있을 뿐입니다.
- **epoch**을 늘리면 처음엔 성능이 오르지만, 계속 늘린다고 좋아지지만은 않습니다 (과적합).
- **optimizer**는 같은 목적지(loss 최소화)로 가는 다른 경로입니다. adamw가 이 프로젝트의 기본값인 이유는
  대체로 무난하게 빠르고 안정적으로 수렴하기 때문입니다.
- **activation**은 층 사이에 비선형성을 주는 장치입니다. 이게 없으면 아무리 층을 쌓아도 결국 하나의
  선형 변환과 같아져 복잡한 패턴을 배울 수 없습니다.
- 배터리 프로젝트 `train_baseline.py`에도 `--optimizer {adamw,adam,sgd}` 옵션이 추가되어 있습니다.
  여기서 확인한 경향을 실제 배터리 이미지 개선 실험에도 적용해볼 수 있습니다.